# SW 4 — per-CR archive and animation

A small, reproducible CR 2203 example. Propagation writes one rounded-core product; the movie below ingests the archive. This sample uses the locally cached 2018 exact-IDL CH-area Parquet, not SQL.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/helio_n_matplotlib")
os.environ.setdefault("SUNPY_CONFIGDIR", "/tmp/helio_n_sunpy")

import numpy as np
import pandas as pd
from IPython.display import Video, display

project_root = Path.cwd().resolve()
if not (project_root / "Library").exists():
    project_root = project_root.parent
assert (project_root / "Library").exists()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from Library.SW.Archive import cr_bounds, load_cr, load_cube, load_inputs, load_series
from Library.SW.Config import load_empirical_spec
from Library.SW.Visualization import plot_polar_snapshot


## 1. Propagate one rotation

The example archive is separate from the future production SQL archive. The command refuses to overwrite an existing CR; re-running this cell reuses it.

In [ ]:
cr = 2203
cr_start, cr_end = cr_bounds(cr)
archive_root = project_root / "Outputs/SW/Samples/Archive"
source_path = project_root / "Outputs/Filaments/from-miracle/CH Areas 20180101-20181231 idl-exact.parquet"
assert source_path.exists(), f"Sample source is missing: {source_path}"

if not (archive_root / f"CR{cr:04d}" / "manifest.json").exists():
    subprocess.run(
        [sys.executable, str(project_root / "Scripts/Make.py"), "propagate_sw", str(cr),
         "--input-source", "parquet", "--input-parquet", str(source_path),
         "--archive-root", str(archive_root)],
        check=True, cwd=project_root,
    )
cr_start, cr_end


## 2. Ingest the dense cube and rounded-core series

`speed` and `is_slow_wind` are separate arrays with named `(time, phi, r)` coordinates. Missing speed stays NaN. `load_series` returns a DataFrame for live analysis; a range can span adjacent CRs once archived. Each astronomical CR boundary is rounded to the nearest output hour, so adjacent products concatenate on one hourly lattice.

In [ ]:
cube = load_cr(archive_root, cr)
series = load_series(archive_root, cr_start, cr_end)
summary = pd.Series({
    "frames": cube.sizes["time"],
    "longitudes": cube.sizes["phi"],
    "shells": cube.sizes["r"],
    "finite_fraction": float(np.isfinite(cube.speed.values).mean()),
    "series_rows": len(series),
})
display(summary)
display(load_inputs(archive_root, cr))
series


In [ ]:
window_start = pd.Timestamp("2018-04-22 00:00")
window_end = window_start + pd.Timedelta(hours=6)
window_cube = load_cube(archive_root, window_start, window_end)
window_series = load_series(archive_root, window_start, window_end)
assert window_cube.sizes["time"] == 6
display(window_series)
window_cube.speed.isel(time=0)


## 3. Inspect a 2D still frame

The frame and satellite panels below are rendered from the archived cube and series; this cell does not invoke the propagator.

In [ ]:
comparison_frames = {
    sat: series["satellite"][sat].copy()
    for sat in series["satellite"].columns.get_level_values(0).unique()
}
for sat, frame in comparison_frames.items():
    frame.attrs["label"] = {"ace_earth": "ACE @ Earth", "stereo_a": "STEREO-A"}.get(sat, sat)
times = pd.DatetimeIndex(cube.time.values)
finite_speed = cube.speed.values[np.isfinite(cube.speed.values)]
plot_polar_snapshot(
    str(window_start), times, cube.phi.values, cube.r.values,
    cube.speed.values, (float(finite_speed.min()), float(finite_speed.max())),
    cube.is_slow_wind.values, load_empirical_spec().slow_sw_speed(times),
    comparison_frames,
)


## 4. Build a short movie from the archive

This command reads saved HDF5/Parquet products only. A single CR number selects CR mode with seven days of padding on each side; it requires the neighboring CR archives. For example: `python Scripts/Make.py make_animation 2203 --archive-root Outputs/SW/Samples/Archive`. The two-timestamp mode below remains useful for short or arbitrary windows. Movie output is derived and never seeds propagation.

In [ ]:
movie = project_root / "Outputs/SW/Samples/SW Animation 20180422 6h.mp4"
subprocess.run(
    [sys.executable, str(project_root / "Scripts/Make.py"), "make_animation",
     str(window_start), str(window_end), "--archive-root", str(archive_root),
     "--output", str(movie), "--fps", "6", "--dpi", "70"],
    check=True, cwd=project_root,
)
display(Video(filename=str(movie), embed=True))
